## A User's Guide / Entry Point To The Concatenator
### Chris Tralie

In [1]:
%%html
<iframe width="560" height="315" src="https://www.youtube.com/embed/yktn9Rwi7CY?si=b1_wyP0bZSMgf1Ie" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

In [ ]:
from scipy.io import wavfile
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
import librosa
import librosa.display
from scipy import sparse
import sys
sys.path.append("../src")
from driedger import *
from particle import *
from probutils import *
from audioutils import *
import time

In [ ]:
sr = 44100 # Audio sample rate
win = 2048 # Window length. A longer window has better pitch resolution but may muddle transients
stereo = False

ytarget = load_corpus("../target/Beatles_LetItBe.mp3", sr=sr, stereo=stereo)[0]
ycorpus, _, start_idxs = load_corpus("../corpus/Bees_Buzzing.mp3", sr=sr, stereo=stereo, hop=win//2)

In [ ]:
tic = time.time()

feature_params = dict(
    win=win,
    sr=sr,
    min_freq=0,
    max_freq=8000
)
particle_params = dict(
    p=5, # A positive number of "active voices" or "polyphony"
    
    pd=0.95, # A number between 0 and 1.  If this is higher, the audio grains will be longer
    
    temperature=10, # A nonnegative number.  If this is higher, we care more about matching the target, 
                    # but it may make grains shorter
    L=10, # Number of iterations for observation fits.  Decrease this to run faster at the expense of accuracy
    
    P=1000, # Number of particles.  Increase this to get better results, at the expense of computation
    
    r=10, # Repeated activation suppression.  Increase this to cut down on jitters
    
    alpha=0.1 # A nonnegative number.  The higher this is, the more consistent amplitude levels are in the output, 
              # but the less we match amplitude fluctuations in the target
)

pf = ParticleAudioProcessor(ycorpus, start_idxs, feature_params, particle_params, device='np')
ygen = pf.process_audio_offline(ytarget)
print("Elapsed time particle filter: {:.3f}".format(time.time()-tic))

In [ ]:
ipd.Audio(ygen.T, rate=sr)